# The Sources

https://medium.com/@azimkhan8018/a-beginners-guide-to-deep-learning-with-mnist-dataset-0894f7183344 - syntax from here

https://www.tensorflow.org/datasets/keras_example - layer details from here

https://huggingface.co/docs/optimum/concept_guides/quantization - quantization from here

[University of Florida PHZ4710 Handwritten Digits Classification](https://cmp.phys.ufl.edu/PHZ4710/files/unit7/mnist.html#train-network) - CNN architecture from here.

This revision keeps the original notebook's optimizer, batch size, validation split and signed INT8 calibration and export flow. AI assistance: Codex.


# Import modules...

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten
import matplotlib.pyplot as plt
import numpy as np
import os

keras.utils.set_random_seed(196)


# Looking at the training and test data...

In [ ]:
# This downloads the mnist dataset from keras
(x_train,y_train),(x_test,y_test) = keras.datasets.mnist.load_data()

print(y_train[67]) # Look, for it is the index 67 entry, and its label (y_train) is 1;
plt.figure()
plt.imshow(x_train[67]) # And the actual image plotted, is indeed 1! (A questionable-looking 1, but it's what it is)

In [ ]:
#Meanwhile the test data for testing accuracy, not training
print(f"Below is the number... {y_test[67]}")
plt.imshow(x_test[67])

#Same thing; (x) data <-> (y) label

# Preprocessing/normalization

In [ ]:
# Normalize 0–255 pixels and keep the height and width for Conv2D.
x_train = x_train.astype("float32")[..., None] / 255.0
x_test = x_test.astype("float32")[..., None] / 255.0

# Constructing model

In [ ]:
# Architecture from the University of Florida notebook.
model = Sequential([
    keras.Input(shape=(28, 28, 1)),
    Conv2D(16, 8, strides=2, padding="valid", activation="relu", name="conv1"),
    Conv2D(8, 5, strides=2, padding="valid", activation="relu", name="conv2"),
    Flatten(),
    Dense(32, activation="relu", name="dense1"),
    Dense(10, activation="softmax", name="dense2"),
])
model.summary()

# Compile and train model!

In [ ]:
# Compile with sparse labels and the published softmax output.
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

# Keep the best validation checkpoints before evaluating on the test set.
checkpoints = [
    keras.callbacks.ModelCheckpoint(
        "best_accuracy.weights.h5", monitor="val_sparse_categorical_accuracy",
        mode="max", save_best_only=True, save_weights_only=True,
    ),
    keras.callbacks.ModelCheckpoint(
        "best_loss.weights.h5", monitor="val_loss",
        mode="min", save_best_only=True, save_weights_only=True,
    ),
]

# Train
history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=40,
    validation_split=0.2, #Uses some of the training data as a validation set
    shuffle=True,
    callbacks=checkpoints,
)

# Evaluate model performance

In [ ]:
best_accuracy_epoch = int(np.argmax(history.history["val_sparse_categorical_accuracy"])) + 1
best_loss_epoch = int(np.argmin(history.history["val_loss"])) + 1
print(f"Best validation accuracy: {history.history['val_sparse_categorical_accuracy'][best_accuracy_epoch-1]:.4f} at epoch {best_accuracy_epoch}")
print(f"Lowest validation loss: {history.history['val_loss'][best_loss_epoch-1]:.6f} at epoch {best_loss_epoch}")

# Use minimum validation loss. It kept accuracy close to the peak on this run.
model.load_weights("best_loss.weights.h5")
y_probs = model.predict(x_test)
y_pred = y_probs.argmax(axis=1)

from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))

# Plot accuracy over epochs


In [ ]:
plt.plot(history.history["sparse_categorical_accuracy"], label="Training")
plt.plot(history.history["val_sparse_categorical_accuracy"], label="Validation")
plt.legend()

# Plot loss over epochs


In [ ]:
plt.plot(history.history['loss'], label="Training")
plt.plot(history.history['val_loss'], label="Validation")
plt.legend()

# Calibration

Here we find the scaling factor assuming...

signed INT8, zero-point 0, one scale per tensor.

Calibrate on the training partition only, using the outputs after ReLU. The
last layer exports its biased INT32 logits for classification.


In [ ]:
def scale(x):
  # Largest magnitude maps to 127 with a symmetric signed INT8 scale.
  peak = float(np.max(np.abs(x)))
  return peak / 127 if peak else 1.0

x_cal = x_train[:48000]  # Exclude the 20% validation partition.
s_input_initial = scale(x_cal)
weighted_layers = [layer for layer in model.layers if isinstance(layer, (Conv2D, Dense))]
relu_outputs = keras.Model(inputs=model.inputs, outputs=[layer.output for layer in weighted_layers[:-1]])

# Accumulate peaks in batches to bound the size of temporary feature maps.
peaks = np.zeros(len(weighted_layers) - 1)
for start in range(0, len(x_cal), 128):
  activations = relu_outputs(x_cal[start:start + 128], training=False)
  peaks = np.maximum(peaks, [np.max(np.abs(value.numpy())) for value in activations])

s_in = s_input_initial
layer_params = []
for i, layer in enumerate(weighted_layers):
  quantize = i != len(weighted_layers) - 1
  has_relu = layer.activation == keras.activations.relu
  s_w = scale(layer.kernel.numpy())
  s_bias = s_in * s_w
  s_out = (peaks[i] / 127 if peaks[i] else 1.0) if quantize else s_bias

  layer_params.append(dict(
      name=layer.name, s_in=s_in, s_w=s_w, s_bias=s_bias, s_out=s_out,
      multiplier=s_bias / s_out, quantize=quantize, has_relu=has_relu,
  ))
  print(f"{layer.name}: weight scale={s_w}, bias scale={s_bias}, output scale={s_out}")
  s_in = s_out

# Converting scaling factor per layer to shift and multiply

We convert the output multiplier per layer to a right shift (r) and unsigned multiply (M). Simply:

output multiplier ~ M/2^r

In [ ]:
def multiplier_and_shift(m):
  for shift in range(31, -1, -1):  # Largest valid shift gives highest precision.
    multiplier = int(round(m * (1 << shift)))
    if 1 <= multiplier <= 0xFFFFFFFF:
      return multiplier, shift
  raise ValueError(f"Unrepresentable scale ratio: {m}")

for p in layer_params:
  m, s = multiplier_and_shift(p["multiplier"]) if p["quantize"] else (1, 0)
  p["q_multiplier"] = m
  p["q_shift"] = s
  print(f"{p['name']}: M={m}, shift={s}")

# Quantization and export

We quantize the weights to be signed INT8, and bias to be signed INT32 and export them.

Weights use accelerator KCRS order and feature maps use CHW order in memory.
Flatten reuses the feature map's address. The first Dense weights are permuted
at export to match that memory order. ReLU is a combinational clamp after
requantization on each completed write.


In [ ]:
export_dict = {"s_input": np.array(s_input_initial, dtype=np.float32)}
for layer, p in zip(weighted_layers, layer_params):
  weights, biases = layer.get_weights()
  if isinstance(layer, Conv2D):
    # Keras RSCK -> accelerator KCRS.
    weights = weights.transpose(3, 2, 0, 1)
    p["stride"] = layer.strides
  else:
    if layer.name == "dense1":
      # Keras flattens HWC; external memory flattens CHW in both implementations.
      h, w, c = tuple(model.get_layer("conv2").output.shape[1:])
      weights = weights.reshape(h, w, c, layer.units).transpose(3, 2, 0, 1)
    else:
      weights = weights.T
    weights = weights.reshape(layer.units, -1, 1, 1)
    p["stride"] = (1, 1)

  p["q_weights"] = np.clip(np.round(weights / p["s_w"]), -127, 127).astype(np.int8)
  q_biases = np.round(biases / p["s_bias"])
  limits = np.iinfo(np.int32)
  if np.any((q_biases < limits.min) | (q_biases > limits.max)):
    raise OverflowError(f"{layer.name} bias exceeds INT32 range")
  p["q_biases"] = q_biases.astype(np.int32)

  name = p["name"]
  export_dict[f"{name}_weights"] = p["q_weights"]
  export_dict[f"{name}_biases"] = p["q_biases"]
  export_dict[f"{name}_multiplier"] = np.array(p["q_multiplier"], dtype=np.uint32)
  export_dict[f"{name}_shift"] = np.array(p["q_shift"], dtype=np.int32)
  export_dict[f"{name}_stride"] = np.array(p["stride"], dtype=np.int32)
  export_dict[f"{name}_quantize"] = np.array(p["quantize"])
  export_dict[f"{name}_relu"] = np.array(p["has_relu"])
  print(f"{name}: weights={p['q_weights'].shape}, biases={p['q_biases'].shape}")

np.savez("mnist_quantized_int8.npz", **export_dict)
model.save("mnist_cnn.keras")
print("Exported quantized parameters to mnist_quantized_int8.npz")

def integer_layer(x, p):
  weights = p["q_weights"].astype(np.int32)
  k, c, r, s = weights.shape
  si, sj = p["stride"]
  if r == s == 1:
    x = x.reshape(len(x), c, 1, 1)  # CHW memory as a single position.
  windows = np.lib.stride_tricks.sliding_window_view(x, (r, s), axis=(2, 3))
  windows = windows[:, :, ::si, ::sj, :, :]
  n, _, h, w, _, _ = windows.shape
  rows = windows.transpose(0, 2, 3, 1, 4, 5).reshape(n, h * w, c * r * s)
  # NumPy signed INT32 matmul and add match the accumulator wrap.
  acc = rows.astype(np.int32) @ weights.reshape(k, -1).T + p["q_biases"]
  output = acc.transpose(0, 2, 1).reshape(n, k, h, w)
  if p["quantize"]:
    product = output.astype(np.int64) * np.int64(p["q_multiplier"])
    shift = p["q_shift"]
    if shift:
      magnitude = (np.abs(product) + (1 << (shift - 1))) >> shift
      product = np.where(product < 0, -magnitude, magnitude)
    output = np.clip(product, -128, 127).astype(np.int8)
  if p["has_relu"]:
    output = np.maximum(output, 0)
  return output

# Verify integer-only inference accuracy and keep actual chained reference tensors.
q_input = np.clip(np.round(x_test / s_input_initial), -127, 127).astype(np.int8)
q_input = q_input.transpose(0, 3, 1, 2)
sample_ids = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 17, 18, 21, 30, 35, 61, 84])  # Same test indices as the dense RTL replay.
samples = {"ids": sample_ids, "labels": y_test[sample_ids], "inputs": q_input[sample_ids]}
scores = []
for start in range(0, len(q_input), 128):
  q_x = q_input[start:start + 128]
  for i, p in enumerate(layer_params):
    q_x = integer_layer(q_x, p)
    if start == 0:
      samples[f"layer{i}"] = q_x[sample_ids]
  scores.append(q_x.reshape(-1, 10))

q_x = np.concatenate(scores)
q_pred = q_x.argmax(axis=1)
q_acc = accuracy_score(y_test, q_pred)
np.savez("rtl_samples.npz", **samples)
print(f"float32 Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Quantized int8 Test Accuracy: {q_acc:.4f}")

# Exporting to Hex

In [ ]:
os.makedirs("hex_export", exist_ok=True)

for p in layer_params:
  name = p["name"]

  #export int8 weights - 2-digit two's complement hex, one value per line
  weights_flat = p["q_weights"].ravel()
  with open(f"hex_export/{name}_weights.hex", "w") as f:
        for w in weights_flat:
            #& 0xFF converts signed int8 (-127..127) to unsigned 8-bit two's complement
            f.write(f"{int(w) & 0xFF:02x}\n")

  #export int32 biases - 8-digit two's complement hex, one value per line
  biases_flat = p["q_biases"].flatten()
  with open(f"hex_export/{name}_biases.hex", "w") as f:
        for b in biases_flat:
            # Mask signed INT32 into its 32-bit two's complement word.
            f.write(f"{int(b) & 0xFFFFFFFF:08x}\n")

  print(f"Exported {name}_weights.hex ({len(weights_flat)} lines) and {name}_biases.hex ({len(biases_flat)} lines)")

#export multipliers and shifts summary
with open("hex_export/quantization_params.txt", "w") as f:
  f.write(f"s_input: {s_input_initial}\n\n")
  for p in layer_params:
      f.write(f"[{p['name']}]\n")
      f.write(f"  multiplier (dec): {p['q_multiplier']}\n")
      f.write(f"  multiplier (hex): 0x{p['q_multiplier']:08x}\n")
      f.write(f"  weight_shape:     {p['q_weights'].shape} (KCRS)\n")
      f.write(f"  stride:           {p['stride']}\n")
      f.write(f"  quantize:         {int(p['quantize'])}\n")
      f.write(f"  relu:             {int(p['has_relu'])}\n")
      f.write(f"  right_shift:      {p['q_shift']}\n\n")

print("\nExported fixed-point multipliers and shifts to 'hex_export/quantization_params.txt'")